In [2]:
print("Looking in:", os.path.abspath(data_path))
print("Files found:", all_files)


Looking in: c:\Users\Nandini\Desktop\python\AI-ML-project\data\raw
Files found: []


In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import joblib
import glob
import os

data_path = "../data/raw/"
all_files = glob.glob(os.path.join(data_path, "*.csv"))

df_list = []
for filename in all_files:
    temp_df = pd.read_csv(filename)
    temp_df.columns = temp_df.columns.str.strip()
    df_list.append(temp_df)

data = pd.concat(df_list, axis=0, ignore_index=True)
print(data.shape)

(2830743, 79)


In [5]:
data.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Rows with NaN:", data.isnull().any(axis=1).sum())

data.dropna(inplace=True)
print("Shape after dropping NaN rows:", data.shape)

Rows with NaN: 2867
Shape after dropping NaN rows: (2827876, 79)


In [6]:
print("Duplicate rows:", data.duplicated().sum())
data.drop_duplicates(inplace=True)
print("Shape after dropping duplicates:", data.shape)

Duplicate rows: 307078
Shape after dropping duplicates: (2520798, 79)


In [7]:
print(data['Label'].unique())


<ArrowStringArray>
[                    'BENIGN',                       'DDoS',
                   'PortScan',                        'Bot',
               'Infiltration',   'Web Attack � Brute Force',
           'Web Attack � XSS', 'Web Attack � Sql Injection',
                'FTP-Patator',                'SSH-Patator',
              'DoS slowloris',           'DoS Slowhttptest',
                   'DoS Hulk',              'DoS GoldenEye',
                 'Heartbleed']
Length: 15, dtype: str


In [8]:
def map_to_category(label):
    label = label.strip()
    if label == 'BENIGN':
        return 'Normal'
    elif label in ['DoS Hulk', 'DoS GoldenEye', 'DoS slowloris', 'DoS Slowhttptest', 'Heartbleed', 'DDoS']:
        return 'DoS'
    elif label in ['PortScan']:
        return 'Probe'
    elif label in ['FTP-Patator', 'SSH-Patator', 'Bot'] or 'Web Attack' in label:
        return 'R2L'
    elif label in ['Infiltration']:
        return 'U2R'
    else:
        return 'Other'

data['Category'] = data['Label'].apply(map_to_category)
print(data['Category'].value_counts())

Category
Normal    2095057
DoS        321770
Probe       90694
R2L         13241
U2R            36
Name: count, dtype: int64


In [9]:
cols_to_drop = ['Label']
data.drop(columns=cols_to_drop, inplace=True)

if 'Fwd Header Length.1' in data.columns:
    data.drop(columns=['Fwd Header Length.1'], inplace=True)

print(data.shape)
print(data.columns.tolist())

(2520798, 78)
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE Flag Count', '

In [10]:
X = data.drop(columns=['Category'])
y = data['Category']

print(X.shape, y.shape)
print(X.dtypes.value_counts())

(2520798, 77) (2520798,)
int64      53
float64    24
Name: count, dtype: int64


In [11]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

label_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print(label_mapping)

joblib.dump(label_encoder, '../models/label_encoder.pkl')

{'DoS': np.int64(0), 'Normal': np.int64(1), 'Probe': np.int64(2), 'R2L': np.int64(3), 'U2R': np.int64(4)}


['../models/label_encoder.pkl']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (2016638, 77)
Test shape: (504160, 77)


In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, '../models/scaler.pkl')

['../models/scaler.pkl']

In [14]:
from imblearn.over_sampling import SMOTE

print("Before SMOTE:", pd.Series(y_train).value_counts())

smote = SMOTE(random_state=42, k_neighbors=3)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print("After SMOTE:", pd.Series(y_train_resampled).value_counts())

Before SMOTE: 1    1676045
0     257416
2      72555
3      10593
4         29
Name: count, dtype: int64
After SMOTE: 1    1676045
0    1676045
2    1676045
3    1676045
4    1676045
Name: count, dtype: int64


In [15]:
np.save('../data/processed/X_train.npy', X_train_resampled)
np.save('../data/processed/X_test.npy', X_test_scaled)
np.save('../data/processed/y_train.npy', y_train_resampled)
np.save('../data/processed/y_test.npy', y_test)

joblib.dump(list(X.columns), '../models/feature_names.pkl')

print("All preprocessed data saved successfully.")

All preprocessed data saved successfully.
